# NeoOLAF Resume From Valid PipelineState

This notebook resumes a NeoOLAF run after a crash, but unlike the previous version it does **not** trust every file named `state.json`.

It tests each candidate with:

```python
PipelineState.load_json(path)
```

and only uses a file that NeoOLAF itself accepts as a valid `PipelineState`.

If no valid Layer 3 state exists, it resumes from the latest valid Layer 2 state and reruns Layer 3 onward.

In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
import shlex
import platform
from datetime import datetime

print("Python executable:", sys.executable)
print("Platform:", platform.platform())
print("Current working directory:", Path.cwd())

In [ ]:
# =========================
# CONFIGURATION
# =========================

# Leave None for automatic detection.
PROJECT_ROOT = None

# If automatic run detection chooses the wrong run, set this manually.
RUN_DIR = None

MODEL = "openrouter/openai/gpt-oss-20b"
PROFILE = "xquality_loose"

FROM_LAYER = 3
TO_LAYER = 12

# Prefer valid Layer 3 PipelineState if available.
# If not available, the notebook automatically falls back to valid Layer 2 PipelineState.
PREFER_LAYER03_STATE = True

USE_SAME_RUN_DIR = True
RESUME_SUFFIX = "_resumed_from_layer03"

MAX_CONCURRENCY_LAYER03 = 4
MAX_CONCURRENCY_LAYER04 = 4
MAX_CONCURRENCY_LAYER05 = 4
MAX_CONCURRENCY_LAYER06 = 4
MAX_CONCURRENCY_LAYER07 = 4
MAX_CONCURRENCY_LAYER08 = 4
MAX_CONCURRENCY_LAYER09 = 4
MAX_CONCURRENCY_LAYER10 = 4
MAX_CONCURRENCY_LAYER11 = 4
MAX_CONCURRENCY_LAYER12 = 4

RAG_BACKEND = None
VERBOSE = True

## 1. Resolve project root

In [ ]:
def looks_like_neoolaf_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    return (
        (path / "src" / "neoolaf").exists()
        or (path / "neoolaf").exists()
        or (path / "pyproject.toml").exists()
    )

def candidate_roots():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    for key in ["NEOOLAF_PROJECT_ROOT", "PROJECT_ROOT"]:
        if os.environ.get(key):
            candidates.append(Path(os.environ[key]).expanduser())
    candidates.extend([
        Path("/home/galencarmedeiro/git/postdoc/NeoOLAF"),
        Path("C:/Users/henri/Documents/git/post-doc/NeoOLAF"),
        Path.home() / "Documents" / "git" / "post-doc" / "NeoOLAF",
        Path.home() / "git" / "postdoc" / "NeoOLAF",
    ])
    seen = set()
    out = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        if str(r).lower() not in seen:
            seen.add(str(r).lower())
            out.append(r)
    return out

if PROJECT_ROOT is None:
    found = [p for p in candidate_roots() if looks_like_neoolaf_root(p)]
    if not found:
        print("Could not find NeoOLAF root. Tried:")
        for p in candidate_roots():
            print(" -", p)
        raise RuntimeError("Set PROJECT_ROOT manually to the NeoOLAF repository path visible to this kernel.")
    PROJECT_ROOT = found[0]
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
    if not looks_like_neoolaf_root(PROJECT_ROOT):
        raise RuntimeError(f"PROJECT_ROOT does not look like NeoOLAF root: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PWD:", Path.cwd())

# Ensure local src is importable for editable and non-editable installs.
src_path = PROJECT_ROOT / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    from neoolaf.core.pipeline_state import PipelineState
    print("Imported PipelineState successfully.")
except Exception as e:
    raise RuntimeError(
        "Could not import neoolaf.core.pipeline_state.PipelineState. "
        "Select the NeoOLAF venv kernel or run Jupyter from the NeoOLAF environment."
    ) from e

## 2. Find and validate candidate state files

This cell lists `state.json` files and marks which ones are valid `PipelineState` files.

In [ ]:
from neoolaf.core.pipeline_state import PipelineState

LAYER02_NAME = "layer02_candidate_enrichment"
LAYER03_NAME = "layer03_candidate_typing_resolution"

def recent_state_files(root: Path, limit: int = 300):
    bases = []
    for name in ["examples", "runs", "."]:
        p = root / name
        if p.exists():
            bases.append(p)

    found = []
    seen = set()
    for base in bases:
        for p in base.rglob("state.json"):
            try:
                rp = p.resolve()
            except Exception:
                rp = p
            if str(rp).lower() in seen:
                continue
            seen.add(str(rp).lower())
            try:
                found.append((p.stat().st_mtime, p))
            except OSError:
                pass
    found.sort(reverse=True)
    return found[:limit]

def is_valid_pipeline_state(path: Path):
    try:
        PipelineState.load_json(str(path))
        return True, ""
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

states = recent_state_files(PROJECT_ROOT, limit=300)
validated = []

for mtime, p in states:
    valid, err = is_valid_pipeline_state(p)
    validated.append({
        "mtime": mtime,
        "path": p,
        "valid": valid,
        "error": err,
        "layer02": LAYER02_NAME in str(p),
        "layer03": LAYER03_NAME in str(p),
    })

print(f"Found {len(validated)} state.json files.")
print("\nRecent Layer 2/3 state candidates:")
shown = 0
for item in validated:
    p = item["path"]
    if not (item["layer02"] or item["layer03"]):
        continue
    shown += 1
    status = "VALID" if item["valid"] else "invalid"
    layer = "L3" if item["layer03"] else "L2"
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except Exception:
        rel = p
    print(f"{shown:02d} | {datetime.fromtimestamp(item['mtime']).strftime('%Y-%m-%d %H:%M:%S')} | {layer} | {status} | {rel}")
    if not item["valid"]:
        print("     ", item["error"])
    if shown >= 80:
        break

valid_l3 = [x for x in validated if x["valid"] and x["layer03"]]
valid_l2 = [x for x in validated if x["valid"] and x["layer02"]]

print("\nValid Layer 3 PipelineStates:", len(valid_l3))
print("Valid Layer 2 PipelineStates:", len(valid_l2))

## 3. Select resume state

This prefers a valid Layer 3 state. If none exists, it uses a valid Layer 2 state.

In [ ]:
def infer_run_dir_from_layer_state(state_path: Path) -> Path:
    # RUN_DIR / layerXX_name / state.json
    return state_path.parent.parent

def score_candidate(item):
    s = str(item["path"]).lower()
    score = item["mtime"]
    bonus = 0
    if LAYER03_NAME in s:
        bonus += 10_000_000
    if LAYER02_NAME in s:
        bonus += 8_000_000
    if "prefix_stop_after_03" in s:
        bonus += 1_000_000
    if "xquality" in s:
        bonus += 100_000
    return score + bonus

if RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR).expanduser().resolve()
    candidates = []
    for item in validated:
        if item["valid"] and str(item["path"]).startswith(str(RUN_DIR)):
            candidates.append(item)
else:
    candidates = [x for x in validated if x["valid"] and (x["layer02"] or x["layer03"])]

if not candidates:
    raise RuntimeError(
        "No valid Layer 2 or Layer 3 PipelineState was found. "
        "The files named state.json may be layer-local summaries, not PipelineState files. "
        "Search for a valid root/pipeline state or rerun from the last valid full checkpoint."
    )

valid_l3_candidates = [x for x in candidates if x["layer03"]]
valid_l2_candidates = [x for x in candidates if x["layer02"]]

if PREFER_LAYER03_STATE and valid_l3_candidates:
    chosen = sorted(valid_l3_candidates, key=score_candidate, reverse=True)[0]
    RESUME_REASON = "Using latest valid Layer 3 PipelineState."
elif valid_l2_candidates:
    chosen = sorted(valid_l2_candidates, key=score_candidate, reverse=True)[0]
    RESUME_REASON = "No valid Layer 3 PipelineState selected, using latest valid Layer 2 PipelineState."
elif valid_l3_candidates:
    chosen = sorted(valid_l3_candidates, key=score_candidate, reverse=True)[0]
    RESUME_REASON = "Using valid Layer 3 PipelineState."
else:
    raise RuntimeError("Could not select a valid Layer 2 or Layer 3 PipelineState.")

RESUME_FROM = chosen["path"]
RUN_DIR = infer_run_dir_from_layer_state(RESUME_FROM)

if USE_SAME_RUN_DIR:
    OUTPUT_RUN_DIR = RUN_DIR
else:
    OUTPUT_RUN_DIR = RUN_DIR.parent / f"{RUN_DIR.name}{RESUME_SUFFIX}"
    OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("RESUME_REASON:", RESUME_REASON)
print("RESUME_FROM:", RESUME_FROM)
print("RUN_DIR:", RUN_DIR)
print("OUTPUT_RUN_DIR:", OUTPUT_RUN_DIR)

## 4. Build command

In [ ]:
cmd = [
    sys.executable, "-m", "neoolaf", "run",
    "--resume-from", str(RESUME_FROM),
    "--run-dir", str(OUTPUT_RUN_DIR),
    "--from-layer", str(FROM_LAYER),
    "--to-layer", str(TO_LAYER),
    "--model", MODEL,
    "--profile", PROFILE,
    "--max-concurrency-layer03", str(MAX_CONCURRENCY_LAYER03),
    "--max-concurrency-layer04", str(MAX_CONCURRENCY_LAYER04),
    "--max-concurrency-layer05", str(MAX_CONCURRENCY_LAYER05),
    "--max-concurrency-layer06", str(MAX_CONCURRENCY_LAYER06),
    "--max-concurrency-layer07", str(MAX_CONCURRENCY_LAYER07),
    "--max-concurrency-layer08", str(MAX_CONCURRENCY_LAYER08),
    "--max-concurrency-layer09", str(MAX_CONCURRENCY_LAYER09),
    "--max-concurrency-layer10", str(MAX_CONCURRENCY_LAYER10),
    "--max-concurrency-layer11", str(MAX_CONCURRENCY_LAYER11),
    "--max-concurrency-layer12", str(MAX_CONCURRENCY_LAYER12),
]

if RAG_BACKEND:
    cmd.extend(["--rag-backend", RAG_BACKEND])

if VERBOSE:
    cmd.append("--verbose")

print("Command:")
print(" ".join(shlex.quote(str(x)) for x in cmd))

## 5. Run resume with visible errors

In [ ]:
env = os.environ.copy()

print("Starting resume at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    env=env,
    check=False,
    capture_output=True,
    text=True,
)

print("Finished with return code:", result.returncode)

print("\n===== STDOUT tail =====")
print(result.stdout[-12000:])

print("\n===== STDERR tail =====")
print(result.stderr[-12000:])

if result.returncode != 0:
    raise RuntimeError("Resume command failed. See STDOUT/STDERR above.")

## 6. Latest states and exports

In [ ]:
states_after = recent_state_files(OUTPUT_RUN_DIR, limit=120)

print(f"Latest states under {OUTPUT_RUN_DIR}:")
for i, (mtime, p) in enumerate(states_after[:80], 1):
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except Exception:
        rel = p
    valid, err = is_valid_pipeline_state(p)
    status = "VALID" if valid else "invalid"
    print(f"{i:02d} | {datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')} | {status} | {rel}")

print("\nPossible Layer 12/export files:")
for p in OUTPUT_RUN_DIR.rglob("*"):
    if p.is_file() and p.name in {
        "kg_local.json",
        "kg_inferred.json",
        "ontology_local.ttl",
        "ontology_inferred.ttl",
        "state.json",
    }:
        s = str(p).lower()
        if "layer12" in s or "exports" in s:
            print(p)